# SoundStream - Simulaciones de Base de Datos con Pandas
**Proyecto Integrador - Nuevas Tecnologias**

**Integrante:** Daniela Bravo

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from utils.simulacion_usuarios import crear_tabla_usuarios, describir_tabla_usuarios
from utils.simulacion_canciones import crear_tabla_canciones, describir_tabla_canciones
from utils.simulacion_listas import crear_tabla_listas, describir_tabla_listas
from utils.simulacion_favoritas import crear_tabla_favoritas, describir_tabla_favoritas
from utils.simulacion_lista_canciones import crear_tabla_lista_canciones, describir_tabla_lista_canciones
from utils.simulacion_artistas import crear_tabla_artistas, describir_tabla_artistas
from utils.simulacion_generos import crear_tabla_generos, describir_tabla_generos

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Tabla: usuarios

In [ ]:
df_usuarios = crear_tabla_usuarios()
print('Estructura:')
print(describir_tabla_usuarios().to_string(index=False))
print(f'\nRegistros: {len(df_usuarios)} | Admins: {(df_usuarios["rol"]=="ADMIN").sum()} | Usuarios: {(df_usuarios["rol"]=="USUARIO").sum()}')
df_usuarios

## 2. Tabla: canciones

In [ ]:
df_canciones = crear_tabla_canciones()
print('Estructura:')
print(describir_tabla_canciones().to_string(index=False))
print(f'\nRegistros: {len(df_canciones)} | Activas: {df_canciones["activo"].sum()} | Reproducciones: {df_canciones["total_reproduccion"].sum():,}')
df_canciones

## 3. Tabla: listas_reproduccion

In [ ]:
df_listas = crear_tabla_listas()
print('Estructura:')
print(describir_tabla_listas().to_string(index=False))
print(f'\nRegistros: {len(df_listas)} | Publicas: {df_listas["es_publica"].sum()} | Privadas: {(~df_listas["es_publica"]).sum()}')
df_listas

## 4. Tabla: canciones_favoritas (Pivote N:M)

In [ ]:
df_favoritas = crear_tabla_favoritas()
print('Estructura:')
print(describir_tabla_favoritas().to_string(index=False))
print(f'\nRegistros: {len(df_favoritas)} | Usuarios con likes: {df_favoritas["id_usuario"].nunique()}')
df_favoritas

## 5. Tabla: lista_canciones (Pivote N:M)

In [ ]:
df_lista_canciones = crear_tabla_lista_canciones()
print('Estructura:')
print(describir_tabla_lista_canciones().to_string(index=False))
print(f'\nRegistros: {len(df_lista_canciones)} | Listas: {df_lista_canciones["id_lista"].nunique()}')
df_lista_canciones

## 6. Tabla: artistas (derivada)

In [ ]:
df_artistas = crear_tabla_artistas(df_canciones)
print('Estructura:')
print(describir_tabla_artistas().to_string(index=False))
print(f'\nTotal artistas: {len(df_artistas)}')
df_artistas.sort_values('total_reproducciones', ascending=False)

## 7. Tabla: generos (derivada)

In [ ]:
df_generos = crear_tabla_generos(df_canciones)
print('Estructura:')
print(describir_tabla_generos().to_string(index=False))
print(f'\nTotal generos: {len(df_generos)}')
df_generos.sort_values('total_reproducciones', ascending=False)

## Resumen General

In [ ]:
resumen = pd.DataFrame({
    'Tabla': ['usuarios', 'canciones', 'listas_reproduccion',
              'canciones_favoritas', 'lista_canciones', 'artistas', 'generos'],
    'Tipo': ['Principal', 'Principal', 'Principal',
             'Pivote (N:M)', 'Pivote (N:M)', 'Derivada', 'Derivada'],
    'Registros': [len(df_usuarios), len(df_canciones), len(df_listas),
                  len(df_favoritas), len(df_lista_canciones),
                  len(df_artistas), len(df_generos)],
    'Columnas': [len(df_usuarios.columns), len(df_canciones.columns), len(df_listas.columns),
                 len(df_favoritas.columns), len(df_lista_canciones.columns),
                 len(df_artistas.columns), len(df_generos.columns)]
})
print(f'Total registros: {resumen["Registros"].sum()} | Total tablas: {len(resumen)}')
resumen

## Consultas de Ejemplo

In [ ]:
# Query 1: Canciones favoritas de maria_music
print('Canciones favoritas de maria_music (id=3):')
df_favoritas[df_favoritas['id_usuario'] == 3].merge(
    df_canciones[['id_canciones', 'titulo', 'nombre_artista', 'nombre_genero']],
    left_on='id_cancion', right_on='id_canciones'
)[['titulo', 'nombre_artista', 'nombre_genero', 'fecha_me_gusta']]

In [ ]:
# Query 2: Contenido de playlist "Mis Favoritas"
print('Contenido de playlist Mis Favoritas (id=1):')
df_lista_canciones[df_lista_canciones['id_lista'] == 1].merge(
    df_canciones[['id_canciones', 'titulo', 'nombre_artista', 'nombre_genero']],
    left_on='id_cancion', right_on='id_canciones'
)[['posicion', 'titulo', 'nombre_artista', 'nombre_genero']].sort_values('posicion')

In [ ]:
# Query 3: Ranking de usuarios mas activos
print('Ranking de usuarios mas activos:')
fav_count = df_favoritas.groupby('id_usuario').size().reset_index(name='total_favoritas')
list_count = df_listas.groupby('id_usuario').size().reset_index(name='total_listas')

actividad = df_usuarios[df_usuarios['rol'] == 'USUARIO'][['id_usuario', 'nombre_usuario']].merge(
    fav_count, on='id_usuario', how='left'
).merge(
    list_count, on='id_usuario', how='left'
).fillna(0)

actividad['total_favoritas'] = actividad['total_favoritas'].astype(int)
actividad['total_listas'] = actividad['total_listas'].astype(int)
actividad['score_actividad'] = actividad['total_favoritas'] + actividad['total_listas'] * 2
actividad.sort_values('score_actividad', ascending=False)